# Phase 2 — Extension tracks

**Paper 1 · Unrecognized organ damage · AI-READI v3.0.0**

Phase 1 established the paper's backbone. Phase 2 casts the wide net: eight
exposure families against measured organ damage, all **exploratory**, all
logged including the nulls. Phase 3 picks the headline set from this record.

| Track | Question | Verdict |
|---|---|---|
| **C** — psychosocial | Does depression track measured damage? *(Aim 2)* | **nerve only** |
| **A** — glycaemia | Does damage track measured control beyond the treatment label? | **yes, and CV adds** |
| **B** — BMI | Does body mass track damage? | **yes, except kidney** |
| **E** — ECG | Does electrical abnormality co-travel with troponin? | **yes** |
| **D** — wearables | One clean pass | weak |
| **F** — access/SDOH | Do barriers explain being unrecognized? | **opposite direction** |

Two rules run through every track. **Age + severity + site adjustment** is the
default, and **Benjamini–Hochberg** is applied within each experiment's adjusted
family — Phase 2 fits several hundred models, and Phase 3 ranks findings off
this log, so a ranking built on raw p-values would promote noise.

Every number here is recomputed through `src/aireadi`, not read back from the
CSVs, and every one is independently re-verified by
`scripts/verify/verify_e2c.py` and `verify_e2_tracks.py`, which rebuild from the
raw files without importing `aireadi` at all.

## Setup

In [ ]:
# Thin-notebook bootstrap, same as Phases 0 and 1: put `src/` and the paper's
# `scripts/` on the path so this notebook calls exactly the code the runners
# call. No cleaning, scoring or threshold logic is defined here.
import sys, pathlib

REPO = pathlib.Path.cwd()
while not (REPO / "src" / "aireadi").exists():
    REPO = REPO.parent
sys.path[:0] = [str(REPO / "src"), str(REPO / "papers/p1-unrecognized-damage/scripts")]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aireadi import associations, azure_io, figures as fg, omop, results, thresholds
import _phase2

fg.style()
RESULTS = results.results_dir("p1")
pd.set_option("display.width", 200)

In [ ]:
df = _phase2.load()
print(f"{len(df):,} participants, {df.shape[1]} columns")

# Phase 2 adds the derived outcome columns. Note `log_acr`: 254 participants have
# a urine albumin of exactly 0 — a value rounded below the 0.01 mg/dL reporting
# floor, not a missing one — and they are 14.5% of the Healthy group against 8.5%
# of Insulin. A bare log() would drop them and flatten every kidney gradient.
pd.DataFrame({
    "measured": [df.acr_mg_g.notna().sum(), df.troponin_t.notna().sum(),
                 df.monofilament_min.notna().sum()],
    "usable after log": [df.log_acr.notna().sum(), df.log_troponin.notna().sum(),
                         df.monofilament_missed.notna().sum()],
    "if zeros dropped": [df.log_acr_positive.notna().sum(), df.log_troponin.notna().sum(),
                         df.monofilament_missed.notna().sum()],
}, index=["kidney (ACR)", "heart (troponin)", "nerve (monofilament)"])

## Why adjustment strengthens things here — `E2.AGE`

Two tracks produced an association that is absent unadjusted and clear once
covariates enter. That reads like fishing unless the mechanism is shown, so it
is measured once rather than re-argued per track.

**Age is a negative confounder for nearly every exposure this paper tests.**
Younger participants score higher on CES-D, carry more weight, walk more; every
damage outcome rises steeply with age. The two paths cancel, so the crude
estimate is biased toward zero.

The consequence for Phase 3: **an unadjusted number in this log is not a
conservative version of the adjusted one**, and must not be read as a lower
bound.

In [ ]:
age_tbl = pd.read_csv(RESULTS / "E2_AGE_suppression.csv")
opposite = age_tbl[age_tbl.opposite_signs]

fig, ax = fg.new_figure(9.4, 4.6)
exposures = (age_tbl.drop_duplicates("exposure")
             .set_index("exposure")[["exposure_label", "r_exposure_age"]])
outcomes = age_tbl.drop_duplicates("outcome").set_index("outcome")["r_outcome_age"]

ax.barh(range(len(exposures)), exposures.r_exposure_age,
        color=[fg.EMPHASIS if v < 0 else fg.ORGAN["heart"] for v in exposures.r_exposure_age],
        edgecolor=fg.SURFACE, height=0.72)
ax.set_yticks(range(len(exposures)))
ax.set_yticklabels(exposures.exposure_label)
ax.axvline(0, color=fg.BASELINE, linewidth=1.0)
ax.set_xlabel("Correlation with age")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
for i, v in enumerate(exposures.r_exposure_age):
    ax.annotate(f"{v:+.2f}", (v, i), xytext=(6 if v > 0 else -6, 0),
                textcoords="offset points", va="center",
                ha="left" if v > 0 else "right", fontsize=8, color=fg.INK_SECONDARY)

fg.finish(fig, "Almost every Phase-2 exposure declines with age",
          f"while every damage outcome rises with it (r = {outcomes.min():+.2f} to "
          f"{outcomes.max():+.2f}). Age adjustment raises the estimate in "
          f"{100*opposite.adjusted_exceeds_crude.mean():.0f}% of the "
          f"{len(opposite)} opposite-sign pairs.",
          "Source: results/E2_AGE_suppression.csv")
fig.savefig(RESULTS / "E2_AGE_figure.png", dpi=200, bbox_inches="tight")
plt.show()

## Track C — the depression aim (`E2C.1`–`E2C.3`)

Aim 2 asked whether depressive symptoms track measured damage. The honest prior
was null: CES-D was flat against glycaemic outcomes in the exploratory phase.

**It is not null, and the entire signal is nerve.** Four models survive FDR —
CES-D continuous and dichotomised, against abnormal-nerve and against insensate
site count — and all four agree. Kidney and heart show nothing.

In [ ]:
sweep = pd.read_csv(RESULTS / "E2C_1_sweep.csv")
adj = sweep[sweep.adjustment == "damage"].copy()
binary = adj[adj.outcome.isin(associations.BINARY_OUTCOMES)
             & adj.exposure.eq("cesd_total")]

fig, ax = fg.new_figure(8.6, 3.8)
fg.forest(ax, [associations.BINARY_OUTCOMES[o] for o in binary.outcome],
          binary.estimate, binary.ci_lo, binary.ci_hi,
          colors=[fg.ORGAN.get(o.replace("abn_", ""), fg.MUTED) for o in binary.outcome],
          significant=[q < 0.05 for q in binary.q])
ax.set_xlabel("Odds ratio per 1 SD of CES-D-10 (age + severity + site adjusted)")
fg.finish(fig, "Depressive symptoms track nerve damage, and only nerve",
          "Filled = survives Benjamini-Hochberg within the adjusted family. "
          "Kidney and heart intervals sit on the line.",
          "Source: results/E2C_1_sweep.csv")
fig.savefig(RESULTS / "E2C_1_figure.png", dpi=200, bbox_inches="tight")
plt.show()

adj[adj.q < 0.05][["exposure_label", "outcome_label", "n", "estimate", "ci_lo", "ci_hi", "q"]]

### The two questions a reviewer will ask

**Why does adjustment strengthen it?** Age, and only age. CES-D falls with age
(r = −0.22) while insensate sites rise with it (r = +0.21).

**Does it turn on the odd monofilament rows?** `CAVEATS.md` requires this check:
14 participants score 0 on both feet and 6 score 0 on one foot and 10 on the
other. Dropping all 20 leaves the association intact.

In [ ]:
robust = pd.read_csv(RESULTS / "E2C_1_nerve_robustness.csv")
nerve = robust[robust.outcome == "abn_nerve"].set_index("check")

fig, ax = fg.new_figure(8.8, 4.2)
fg.forest(ax, list(nerve.index), nerve.estimate, nerve.ci_lo, nerve.ci_hi,
          colors=[fg.DEEMPHASIS if c == "unadjusted" else fg.ORGAN["nerve"]
                  for c in nerve.index],
          significant=[p < 0.05 for p in nerve.p])
ax.set_xlabel("Odds ratio per 1 SD of CES-D-10, nerve abnormal")
fg.finish(fig, "Age is the suppressor, and the odd feet are not the cause",
          "Adding age alone moves the estimate from null to clear. Excluding all 20 "
          "clinically odd monofilament rows leaves it standing.",
          "Source: results/E2C_1_nerve_robustness.csv")
fig.savefig(RESULTS / "E2C_1_robustness_figure.png", dpi=200, bbox_inches="tight")
plt.show()

### Is it depression, or distress about having diabetes? — `E2C.3`

PAID-5 measures diabetes-specific distress. On the **identical sample**, CES-D
holds and PAID-5 is flat; mutually adjusted, CES-D strengthens and PAID-5 does
not move. The nerve signal is specific to general depressive symptoms.

`E2C.2` asked the reverse question — are depressed participants more likely to be
*unrecognized*? **H3 is null**, and the direction runs opposite to the
hypothesis.

In [ ]:
head = pd.read_csv(RESULTS / "E2C_3_nerve_head_to_head.csv")
h = head[head.outcome == "abn_nerve"].set_index("questionnaire")

fig, ax = fg.new_figure(8.8, 3.6)
fg.forest(ax, list(h.index), h.estimate, h.ci_lo, h.ci_hi,
          colors=[fg.ORGAN["nerve"] if "cesd" in q.lower() or "CES-D" in q else fg.MUTED
                  for q in h.index],
          significant=[p < 0.05 for p in h.p])
ax.set_xlabel("Odds ratio, nerve abnormal (identical complete-case sample)")
fg.finish(fig, "The nerve signal is depression, not diabetes distress",
          f"Both questionnaires fitted on the same n = {int(h.n.iloc[0]):,}. "
          "Mutually adjusted, CES-D strengthens and PAID-5 stays flat.",
          "Source: results/E2C_3_nerve_head_to_head.csv")
fig.savefig(RESULTS / "E2C_3_figure.png", dpi=200, bbox_inches="tight")
plt.show()

## Track A — glycaemia (`E2A.1`, `E2A.2`)

The severity groups are *treatment* categories, not measurements. Does damage
track measured glucose beyond the label?

Yes, strongly, for HbA1c. The more interesting question is whether the expensive
CGM metrics add anything to a cheap blood draw — this is the experiment that
required parsing 2,245 Dexcom streams.

In [ ]:
inc = pd.read_csv(RESULTS / "E2A_1_incremental.csv")
inc["label"] = inc.exposure.map({"tar_180": "TAR > 180", "glucose_cv": "CV",
                                 "mage": "MAGE", "hba1c": "HbA1c"})
piv = inc.pivot(index="label", columns="outcome", values="q_with_mean_glucose")
piv = piv.reindex(["HbA1c", "TAR > 180", "MAGE", "CV"])[list(associations.BINARY_OUTCOMES)]

fig, ax = fg.new_figure(8.6, 3.4)
im = ax.imshow(-np.log10(piv.values), cmap="Blues", vmin=0, vmax=5, aspect="auto")
ax.set_xticks(range(piv.shape[1]))
ax.set_xticklabels([associations.BINARY_OUTCOMES[c].split(" (")[0] for c in piv.columns],
                   rotation=20, ha="right")
ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        q = piv.values[i, j]
        ax.text(j, i, "n.s." if q >= 0.05 else f"q={q:.3g}", ha="center", va="center",
                fontsize=8, color=fg.SURFACE if -np.log10(q) > 2.5 else fg.INK)
ax.grid(False)
fg.finish(fig, "Only glycaemic variability adds anything to mean glucose",
          "Each metric fitted WITH mean glucose already in the model, on one identical "
          "sample. TAR and MAGE add nothing; CV independently predicts kidney damage.",
          "Source: results/E2A_1_incremental.csv")
fig.savefig(RESULTS / "E2A_1_figure.png", dpi=200, bbox_inches="tight")
plt.show()

### Damage when the glucose disagrees with the label — `E2A.2`

46 participants carry **no diabetes label but a diabetes-range HbA1c** (≥ 6.5%).
Against their concordant peers in the same Healthy + Pre-DM universe they have
**four times the odds of kidney damage**. The CGM definition replicates it
independently.

The insulin group at target shows nothing — reaching a glycaemic target does not
undo accumulated damage.

This is the tightest link Phase 2 has to Aim 1: these are people whose *diabetes*
is unrecognized, carrying unrecognized organ damage on top.

In [ ]:
disc = pd.read_csv(RESULTS / "E2A_2_models.csv")
und = disc[disc.definition == "undiagnosed_range"].set_index("outcome")
labels = {"abn_kidney": "Kidney", "abn_heart": "Heart", "abn_nerve": "Nerve",
          "abn_any": "Any organ", "abn_multi": "Two or more"}

fig, ax = fg.new_figure(8.6, 3.6)
fg.forest(ax, [labels[o] for o in und.index], und.estimate, und.ci_lo, und.ci_hi,
          colors=[fg.ORGAN.get(o.replace("abn_", ""), fg.MUTED) for o in und.index],
          significant=[q < 0.05 for q in und.q])
# Bootstrap intervals drawn behind, since every discordant cell is under 50.
for yi, (_, r) in zip(range(len(und))[::-1], und.iterrows()):
    if np.isfinite(r.boot_ci_lo):
        ax.plot([r.boot_ci_lo, r.boot_ci_hi], [yi - 0.22, yi - 0.22],
                color=fg.MUTED, linewidth=1.2, solid_capstyle="round")
ax.set_xlabel("Odds of damage vs concordant peers (age + site adjusted)")
fg.finish(fig, "Undiagnosed-range glycaemia carries four times the kidney damage",
          f"n = {int(und.n_discordant.iloc[0])} with no diabetes label and HbA1c >= 6.5%. "
          "Thin grey bars are bootstrap intervals — every cell is under 50.",
          "Source: results/E2A_2_models.csv")
fig.savefig(RESULTS / "E2A_2_figure.png", dpi=200, bbox_inches="tight")
plt.show()

## Tracks B and E — BMI and the ECG

**BMI** tracks heart, nerve and any-organ damage — but **not kidney at all**
(OR 1.03, p = 0.70). That organ-specific null is worth as much as the positives.

**The ECG's numeric metrics corroborate the troponin signal**: QRS duration
tracks log troponin at p ≈ 5e-25. These are instrument measurements, not machine
interpretations, so they carry no unadjudicated caveat — unlike `E2E.1`.

In [ ]:
bmi = pd.read_csv(RESULTS / "E2B_1_sweep.csv")
b = bmi[(bmi.adjustment == "damage") & (bmi.exposure == "bmi")
        & bmi.outcome.isin(associations.BINARY_OUTCOMES)]
coh = pd.read_csv(RESULTS / "E2E_2_coherence.csv")
c = coh[coh.outcome == "abn_heart"]

fig, axes = plt.subplots(1, 2, figsize=(12.4, 3.6))
for a in axes: a.grid(axis="x", visible=False)

fg.forest(axes[0], [associations.BINARY_OUTCOMES[o].split(" (")[0] for o in b.outcome],
          b.estimate, b.ci_lo, b.ci_hi,
          colors=[fg.ORGAN.get(o.replace("abn_", ""), fg.MUTED) for o in b.outcome],
          significant=[q < 0.05 for q in b.q])
axes[0].set_xlabel("OR per 1 SD of BMI")
axes[0].set_title("BMI: everything except kidney", fontsize=10, loc="left", color=fg.INK)

fg.forest(axes[1], list(c.metric), c.estimate, c.ci_lo, c.ci_hi,
          colors=fg.ORGAN["heart"], significant=[q < 0.05 for q in c.q])
axes[1].set_xlabel("OR per 1 SD, abnormal troponin")
axes[1].set_title("ECG metrics corroborate the heart marker", fontsize=10,
                  loc="left", color=fg.INK)

fg.finish(fig, "BMI is organ-specific; the ECG backs up the troponin",
          "Both age + severity + site adjusted. Hollow markers cross 1.",
          "Source: results/E2B_1_sweep.csv, results/E2E_2_coherence.csv")
fig.savefig(RESULTS / "E2B_E2E_figure.png", dpi=200, bbox_inches="tight")
plt.show()

### `E2E.1` — machine-read prior infarct. **UNADJUDICATED.**

Every one of the 2,251 ECG records carries an explicit *"Unconfirmed Diagnosis"*
stamp. These are waveform-pattern statements generated by the recording device
and never read by a physician, and the standing decision is that this is **one
supplementary row, labelled unadjudicated everywhere it appears, figures
included.**

179 participants carry an unhedged prior-infarct pattern; 149 of the 178 who
answered the item (83.7%) reported no heart attack. The pattern does co-travel
with troponin — 32.8% abnormal against 18.8% in those with no pattern — which
argues it is not pure noise, but it remains a machine reading.

In [ ]:
tiers = pd.read_csv(RESULTS / "E2E_1_tiers.csv").set_index("tier")
by_tier = pd.read_csv(RESULTS / "E2E_1_by_tier.csv").set_index("infarct_tier")
order = ["none", "consider prior infarct", "probable prior infarct",
         "definite prior infarct", "acute or recent only"]
by_tier = by_tier.reindex([o for o in order if o in by_tier.index])

fig, ax = fg.new_figure(9.0, 3.8)
rects = ax.bar(range(len(by_tier)), by_tier.pct_troponin_abnormal,
               color=[fg.DEEMPHASIS if t == "none" else fg.ORGAN["heart"]
                      for t in by_tier.index],
               edgecolor=fg.SURFACE, width=0.62)
for r, (t, row) in zip(rects, by_tier.iterrows()):
    ax.annotate(f"{row.pct_troponin_abnormal:.1f}%\nn={int(row.n)}",
                (r.get_x() + r.get_width()/2, r.get_height()), xytext=(0, 3),
                textcoords="offset points", ha="center", va="bottom",
                fontsize=8, color=fg.INK_SECONDARY)
ax.set_xticks(range(len(by_tier)))
ax.set_xticklabels([t.replace(" prior infarct", "").replace(" ", "\n") for t in by_tier.index])
ax.set_ylabel("% with abnormal troponin")
fg.finish(fig, "UNADJUDICATED — machine-read infarct patterns vs troponin",
          "Machine-generated, physician-unreviewed: every record carries an explicit "
          "'Unconfirmed Diagnosis' stamp. Supplementary only, never a headline.",
          "Source: results/E2E_1_by_tier.csv")
fig.savefig(RESULTS / "E2E_1_figure.png", dpi=200, bbox_inches="tight")
plt.show()

## Tracks D and F — wearables and access barriers

**Track D got one pass, by standing decision.** It also turned up a data defect:
AI-READI computed the Garmin manifest averages *with* the device error codes
included, so a contaminated mean lands between the sentinel and the truth and
sails through any `!= 0` test — 12 resting heart rates under 30 bpm, 113 negative
stress scores on a 0–100 scale. Three of 40 conclusions change once bounds are
applied.

**Track F tested the paper's most attractive explanation** for why damage goes
unrecognized: barriers to care. The one surviving association runs **opposite** to
that hypothesis — more access barriers means *less* likely to be unrecognized —
and it holds within every severity group. The same direction appears in `E2C.2`.
The likely reading is treatment burden: people already in the system for heart
disease report more barriers *because* they are engaging with care.

In [ ]:
f_models = pd.read_csv(RESULTS / "E2F_1_models.csv")
heart = f_models[(f_models.adjustment == "full") & (f_models.outcome == "unrec_heart")]
strata = pd.read_csv(RESULTS / "E2F_1_by_severity.csv")
sh = strata[strata.outcome == "unrec_heart"]

fig, axes = plt.subplots(1, 2, figsize=(12.4, 3.6))
for a in axes: a.grid(axis="x", visible=False)

fg.forest(axes[0], list(heart.exposure_label), heart.estimate, heart.ci_lo, heart.ci_hi,
          colors=fg.ORGAN["heart"], significant=[q < 0.05 for q in heart.q])
axes[0].set_xlabel("OR of being unrecognized — heart")
axes[0].set_title("Fully adjusted, all participants", fontsize=10, loc="left", color=fg.INK)

fg.forest(axes[1], list(sh.stratum), sh.estimate, sh.ci_lo, sh.ci_hi,
          colors=fg.SEVERITY, significant=[p < 0.05 for p in sh.p])
axes[1].set_xlabel("OR per 1 SD of access barriers")
axes[1].set_title("Access barriers, within severity group", fontsize=10,
                  loc="left", color=fg.INK)

fg.finish(fig, "Access barriers predict being told, not being missed",
          "Every estimate sits below 1 — the opposite of the falling-through-the-cracks "
          "hypothesis — and the direction holds in all four severity groups.",
          "Source: results/E2F_1_models.csv, results/E2F_1_by_severity.csv")
fig.savefig(RESULTS / "E2F_1_figure.png", dpi=200, bbox_inches="tight")
plt.show()

### Did the SDOH scoring work?

This is the track with history: the deleted EDA notebooks built three of four
SDOH variables by positionally slicing the *racial discrimination* battery, and
every result from that era is an artifact. (It was never published — this is a
first look, **not** a correction of a field finding.)

Three fresh traps turned up in the scoring, all now in `CAVEATS.md`: two
batteries are **non-monotonic in their coded values** (`pxhi1` runs 0 = no steady
place, 1 = steady, 2 = at risk), three items are skip-gated, and two are nominal.

The check that the scoring is sound: every score rises monotonically across
severity. A slicing bug produces noise, not a clean gradient.

In [ ]:
by_group = pd.read_csv(RESULTS / "E2F_1_by_group.csv").set_index("study_group_label")
show = by_group.loc[fg.SEVERITY_ORDER,
                    ["healthcare_access_barriers", "prescription_unaffordable",
                     "food_insecurity", "housing_insecure"]]

fig, ax = fg.new_figure(9.0, 4.0)
fg.grouped_bars(ax, fg.SEVERITY_ORDER,
                {"Access barriers (0-3)": show.healthcare_access_barriers,
                 "Prescription unaffordable (0-4)": show.prescription_unaffordable,
                 "Food insecurity (0-5)": show.food_insecurity,
                 "Housing insecure (0-1)": show.housing_insecure},
                [fg.ORGAN["kidney"], fg.ORGAN["heart"], fg.ORGAN["nerve"], fg.MUTED],
                fmt="{:.2f}")
ax.set_ylabel("Mean score")
ax.legend(frameon=False, fontsize=8, ncol=2)
fg.finish(fig, "Every hardship score rises across the severity spectrum",
          "The scoring sanity check: a positional-slicing bug produces noise, not four "
          "monotonic gradients.",
          "Source: results/E2F_1_by_group.csv")
fig.savefig(RESULTS / "E2F_1_scoring_figure.png", dpi=200, bbox_inches="tight")
plt.show()

## Where Phase 2 leaves the paper

Nothing here displaces Aim 1. What it adds, in the order Phase 3 should weigh it:

1. **Aim 2 has a result, and it is organ-specific.** Depressive symptoms track
   *nerve* damage — the organ whose damage a patient can actually feel — and not
   the two silent organs. That is a coherent story rather than a lone p-value,
   and it survives dropping every clinically odd row. It is also specific to
   depression, not diabetes distress.
2. **Undiagnosed-range glycaemia carries four times the kidney damage.** The
   tightest link to Aim 1: unrecognized diabetes underneath unrecognized damage.
3. **The ECG corroborates the troponin marker**, which strengthens Aim 1's heart
   half independently of any new claim.
4. **Access barriers do not explain being unrecognized** — the direction is
   opposite, in every severity group. A clean, useful negative.
5. **CV is the only CGM metric that earns its cost** over a cheap HbA1c.

Three data defects were found and fixed, none touching Phase 1: the urine-albumin
reporting floor, the contaminated Garmin averages, and the Dexcom sentinel
strings. All five Phase-1 verifiers still pass.

**Still open for Phase 3.** The nerve finding is cross-sectional and the
direction is genuinely ambiguous — painful neuropathy plausibly causes low mood,
and low mood plausibly reduces foot care. The paper must say so.